# **III. Análisis de identificadores: pacientes, hospitales y médicos**

## Objetivo
Este notebook busca evaluar el rol de las variables identificadoras (`CIP_ENCRIPTADO`/`ID_BENEFICIARIO`, `COD_HOSPITAL`, `MEDICOINTERV1` y `MEDICOALTA`) dentro del modelado predictivo para determinar su viabilidad como características de entrada en los algoritmos de Machine Learning.

## Proceso
1. **Análisis de pacientes repetidos**: Cuantifica la frecuencia de episodios por paciente único a lo largo de los años para evaluar la necesidad de agrupar o descartar el ID.
2. **Análisis de distribución de hospitales**: Analiza la concentración de episodios por hospital (`COD_HOSPITAL`) para evaluar su cardinalidad e impacto territorial.
3. **Análisis de médicos intervinientes y de alta**: Evalúa la viabilidad de mantener la trazabilidad de los profesionales tratantes, observando su cardinalidad y el riesgo potencial de fuga de información (*Data Leakage*).

#### Identificador de pacientes repetidos: CIP_ENCRIPTADO o ID_BENEFICIARIO

1. **CIP_ENCRIPTADO (o ID_BENEFICIARIO)**
- **Tipo de variable:** Demográfica (identificador del paciente).
- **Descripción:** Código identificador único (encriptado) de cada paciente que ingresa al sistema hospitalario.
- **Veredicto:** Descartar. Un número de identificación no posee relación clínica ni operativa con la criticidad del paciente. El análisis exploratorio reveló la existencia de "pacientes" con volúmenes de ingreso humanamente imposibles (como 8.307 episodios en 2019 para el registro "SIN INFORMACIÓN", o 523 episodios para el ID `95162030` en 2022). Esto demuestra que la variable está contaminada con RUTs genéricos o códigos institucionales por defecto para pacientes no enrolados. Aplicar transformaciones como One-Hot Encoding a una variable con cientos de miles de categorías crearía un conjunto de datos propenso al sobreajuste (*overfitting*), afectando el rendimiento de los selectores de características. Cada episodio será evaluado como una instancia independiente según su contexto clínico.

In [1]:
import pandas as pd
import os
import warnings

# Suprimir warnings de DtypeWarning
warnings.simplefilter(action='ignore', category=pd.errors.DtypeWarning)

# ============================================================================
# CONFIGURACIÓN INICIAL: Rutas y archivos
# ============================================================================
carpeta = "../../Datos/Datos clasificados"
ruta_resultados = "../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos"
os.makedirs(ruta_resultados, exist_ok=True)

archivos = [
    "GRD_CLASIFICADO_2019.csv",
    "GRD_CLASIFICADO_2020.csv",
    "GRD_CLASIFICADO_2021.csv",
    "GRD_CLASIFICADO_2022.csv",
    "GRD_CLASIFICADO_2023.csv",
    "GRD_CLASIFICADO_2024.csv"
]

# Mapeo para estandarizar nombres inconsistentes (BOM y cambio de formato 2024)
mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"
}

In [2]:
# ============================================================================
# SECCIÓN 1: ANÁLISIS DE PACIENTES REPETIDOS (CIP_ENCRIPTADO)
# ============================================================================
top5_repetidos_por_año = {}

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    año = archivo[-8:-4]
    
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    if "CIP_ENCRIPTADO" in df.columns:
        df["CIP_ENCRIPTADO"] = df["CIP_ENCRIPTADO"].astype(str)
        df = df[~df["CIP_ENCRIPTADO"].isin(["SIN INFORMACION", "DESCONOCIDO"])]
        
        conteo = df["CIP_ENCRIPTADO"].value_counts()
        repetidos = conteo[conteo > 1]
        
        repetidos_df = repetidos.reset_index()
        repetidos_df.columns = ["CIP_ENCRIPTADO", "count"]
        
        ruta_salida = f"{ruta_resultados}/repetidos_{año}.csv"
        repetidos_df.to_csv(ruta_salida, index=False)
        top5_repetidos_por_año[año] = repetidos_df.head(5)
        
        print(f"✓ {año}: {len(repetidos_df)} pacientes con múltiples episodios. "
              f"Max: {repetidos.max()} episodios. Guardado en {ruta_salida}")

✓ 2019: 165128 pacientes con múltiples episodios. Max: 8307 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2019.csv
✓ 2020: 98379 pacientes con múltiples episodios. Max: 1893 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2020.csv
✓ 2021: 108690 pacientes con múltiples episodios. Max: 2044 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2021.csv
✓ 2022: 129107 pacientes con múltiples episodios. Max: 523 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2022.csv
✓ 2023: 151694 pacientes con múltiples episodios. Max: 341 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2023.csv
✓ 2024: 161533 pacientes con múltiples episodios. Max: 140 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2024.csv


In [3]:
# Pacientes repetidos entre 2019 y 2021
print("="*70)
print("TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2019-2021)")
print("="*70)
for año in ["2019", "2020", "2021"]:
    print(f"\n- Año {año} (Máximo de episodios en {año}: {top5_repetidos_por_año[año]['count'].iloc[0]}):")
    print(top5_repetidos_por_año[año].to_string(index=False))

TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2019-2021)

- Año 2019 (Máximo de episodios en 2019: 8307):
  CIP_ENCRIPTADO  count
SIN INFORMACIÃN   8307
          608828    321
          317005    175
          363057    157
          840781    155

- Año 2020 (Máximo de episodios en 2020: 1893):
  CIP_ENCRIPTADO  count
SIN INFORMACIÃN   1893
         1322640    445
          608828    362
         1099683    279
           24929     79

- Año 2021 (Máximo de episodios en 2021: 2044):
CIP_ENCRIPTADO  count
           nan   2044
    95162030.0    294
    78492052.0    273
    96754933.0    103
    77066145.0     73


In [4]:
# Pacientes repetidos entre 2022 y 2024
print("="*70)
print("TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2022-2024)")
print("="*70)
for año in ["2022", "2023", "2024"]:
    print(f"\n- Año {año} (Máximo de episodios en {año}: {top5_repetidos_por_año[año]['count'].iloc[0]}):")
    print(top5_repetidos_por_año[año].to_string(index=False))

TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2022-2024)

- Año 2022 (Máximo de episodios en 2022: 523):
CIP_ENCRIPTADO  count
      95162030    523
      78492052    302
      96754933    272
      98437046    125
      97796953     83

- Año 2023 (Máximo de episodios en 2023: 341):
CIP_ENCRIPTADO  count
      95162030    341
      78492052    334
      95701049     67
      73381193     55
      78875009     54

- Año 2024 (Máximo de episodios en 2024: 140):
CIP_ENCRIPTADO  count
      78492052    140
      92276704     99
      77008364     79
      95162030     70
      78710945     63


In [5]:
# Generar análisis de tendencias temporal
print("\n" + "="*70)
print("ANÁLISIS DE TENDENCIAS TEMPORALES (Mayor repetición por año)")
print("="*70)
for año in ["2019", "2020", "2021", "2022", "2023", "2024"]:
    max_rep = top5_repetidos_por_año[año]['count'].iloc[0]
    print(f"  {año}: {max_rep:3d} episodios (paciente con mayor repetición)")


ANÁLISIS DE TENDENCIAS TEMPORALES (Mayor repetición por año)
  2019: 8307 episodios (paciente con mayor repetición)
  2020: 1893 episodios (paciente con mayor repetición)
  2021: 2044 episodios (paciente con mayor repetición)
  2022: 523 episodios (paciente con mayor repetición)
  2023: 341 episodios (paciente con mayor repetición)
  2024: 140 episodios (paciente con mayor repetición)


#### Distribución de hospitales en los episodios oncológicos (COD_HOSPITAL)

2. **COD_HOSPITAL**
- **Tipo de variable:** Hospitalaria.
- **Descripción:** Código numérico que identifica al establecimiento de salud específico donde ocurrió el episodio y el egreso del paciente.
- **Veredicto:** Descartar. Tener entre 65 y 72 categorías distintas añade una dimensionalidad muy alta. Además, esta variable está anidada jerárquicamente dentro de `SERVICIO_SALUD` (que agrupa por macrozonas geográficas). Mantener ambas generaría colinealidad. El propósito central de esta propuesta es apoyar la planificación sanitaria y la gestión de recursos a nivel macro (redes de salud), dirigidos a instituciones como FONASA y MINSAL, por lo que el análisis a nivel de hospital individual escapa al alcance epidemiológico de la caracterización.

In [6]:
# ============================================================================
# SECCIÓN 2: ANÁLISIS DE DISTRIBUCIÓN DE HOSPITALES (COD_HOSPITAL)
# ============================================================================

ruta_resultados_hospitales = "../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales"
os.makedirs(ruta_resultados_hospitales, exist_ok=True)

top5_hospitales_por_año = {}
resumen_hospitales = {} # Para optimizar el print final sin releer CSVs

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    año = archivo[-8:-4]
    
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    if "COD_HOSPITAL" in df.columns:
        df["COD_HOSPITAL"] = df["COD_HOSPITAL"].astype(str)
        conteo_hosp = df["COD_HOSPITAL"].value_counts()
        
        hospitales_df = conteo_hosp.reset_index()
        hospitales_df.columns = ["COD_HOSPITAL", "count"]
        
        ruta_salida_hosp = f"{ruta_resultados_hospitales}/hospitales_{año}.csv"
        hospitales_df.to_csv(ruta_salida_hosp, index=False)
        top5_hospitales_por_año[año] = hospitales_df.head(5)
        
        num_hospitales = hospitales_df['COD_HOSPITAL'].nunique()
        top5_volume = hospitales_df.head(5)['count'].sum()
        total_volume = hospitales_df['count'].sum()
        concentracion = (top5_volume / total_volume) * 100
        
        # Guardar en memoria para el resumen final
        resumen_hospitales[año] = (num_hospitales, concentracion)
        
        print(f"✓ {año}: {num_hospitales} hospitales distintos. "
              f"Top 5 concentra {concentracion:.1f}% de episodios. Guardado en {ruta_salida_hosp}")

✓ 2019: 65 hospitales distintos. Top 5 concentra 18.1% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2019.csv
✓ 2020: 65 hospitales distintos. Top 5 concentra 18.0% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2020.csv
✓ 2021: 65 hospitales distintos. Top 5 concentra 18.6% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2021.csv
✓ 2022: 65 hospitales distintos. Top 5 concentra 17.8% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2022.csv
✓ 2023: 68 hospitales distintos. Top 5 concentra 17.0% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2023.csv
✓ 2024: 72 hospitales distintos. Top 5 concentra 16.5% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/

In [8]:
# Hospitales con más episodios entre 2019 y 2021
print("\n" + "="*70)
print("TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2019-2021)")
print("="*70)
for año in ["2019", "2020", "2021"]:
    max_hosp = top5_hospitales_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Hospital con mayor cantidad de episodios: {max_hosp}):")
    print(top5_hospitales_por_año[año].to_string(index=False))


TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2019-2021)

Año 2019 (Hospital con mayor cantidad de episodios: 55726):
COD_HOSPITAL  count
      114101  55726
      106100  42026
      118100  40134
      109100  35921
      121109  35091

Año 2020 (Hospital con mayor cantidad de episodios: 37040):
COD_HOSPITAL  count
      114101  37040
      113100  28652
      116105  25321
      109100  24903
      118100  24660

Año 2021 (Hospital con mayor cantidad de episodios: 44842):
COD_HOSPITAL  count
      114101  44842
      118100  28853
      113100  27955
      116105  25610
      109100  24947


In [9]:
# Hospitales con más episodios entre 2022 y 2024
print("\n" + "="*70)
print("TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2022-2024)")
print("="*70)
for año in ["2022", "2023", "2024"]:
    max_hosp = top5_hospitales_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Hospital con mayor cantidad de episodios: {max_hosp}):")
    print(top5_hospitales_por_año[año].to_string(index=False))


TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2022-2024)

Año 2022 (Hospital con mayor cantidad de episodios: 47426):
COD_HOSPITAL  count
      114101  47426
      118100  31167
      116105  30292
      120101  29101
      109100  28275

Año 2023 (Hospital con mayor cantidad de episodios: 49653):
COD_HOSPITAL  count
      114101  49653
      118100  34144
      116105  32148
      120101  31251
      121109  29786

Año 2024 (Hospital con mayor cantidad de episodios: 49411):
COD_HOSPITAL  count
      114101  49411
      118100  35148
      116105  33858
      120101  30527
      124105  30453


In [10]:
print("\nANÁLISIS DE TENDENCIAS TEMPORAL PARA HOSPITALES")
for año in ["2019", "2020", "2021", "2022", "2023", "2024"]:
    num_hosp, conc_pct = resumen_hospitales[año]
    print(f"  {año}: {num_hosp:2d} hospitales | Concentración (top 5): {conc_pct:5.1f}%")


ANÁLISIS DE TENDENCIAS TEMPORAL PARA HOSPITALES
  2019: 65 hospitales | Concentración (top 5):  18.1%
  2020: 65 hospitales | Concentración (top 5):  18.0%
  2021: 65 hospitales | Concentración (top 5):  18.6%
  2022: 65 hospitales | Concentración (top 5):  17.8%
  2023: 68 hospitales | Concentración (top 5):  17.0%
  2024: 72 hospitales | Concentración (top 5):  16.5%


#### Distribución de médicos (a través de sus identificadores)

3. **MEDICOINTERV1_ENCRIPTADO**
- **Tipo de variable:** Hospitalaria (identificador).
- **Descripción:** Código encriptado que identifica de forma única al médico responsable de realizar la primera intervención o procedimiento quirúrgico.
- **Veredicto:** Descartar. El uso de este identificador es inviable metodológicamente. Registra hasta 11.164 valores únicos por año, generando una matriz dispersa que penalizaría a los algoritmos de selección de características. Además, se detectaron códigos institucionales genéricos que distorsionan la muestra (ej. el ID `98438434.0` registra casi 33.000 intervenciones solo en 2022). Finalmente, existe un quiebre estructural en la codificación de FONASA/MINSAL a partir de 2021 (pasando de IDs numéricos cortos a numeraciones extensas), impidiendo cualquier trazabilidad temporal. El objetivo es caracterizar *qué* procedimientos se hicieron (los cuales se incluirán en el modelado), no *quién* los hizo.

In [11]:
# ============================================================================
# SECCIÓN 3: ANÁLISIS DE MÉDICOS INTERVINIENTES (MEDICOINTERV1_ENCRIPTADO)
# ============================================================================
ruta_resultados_med_interv = "../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención"
os.makedirs(ruta_resultados_med_interv, exist_ok=True)

top5_med_interv_por_año = {}

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    año = archivo[-8:-4]
    
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    if "MEDICOINTERV1_ENCRIPTADO" in df.columns:
        df["MEDICOINTERV1_ENCRIPTADO"] = df["MEDICOINTERV1_ENCRIPTADO"].astype(str)
        df = df[~df["MEDICOINTERV1_ENCRIPTADO"].isin(["SIN INFORMACION", "DESCONOCIDO", "nan", "None"])]
        
        conteo_med = df["MEDICOINTERV1_ENCRIPTADO"].value_counts()
        med_df = conteo_med.reset_index()
        med_df.columns = ["MEDICOINTERV1_ENCRIPTADO", "count"]
        
        ruta_salida = f"{ruta_resultados_med_interv}/medicos_intervencion_{año}.csv"
        med_df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
        top5_med_interv_por_año[año] = med_df.head(5)
        
        num_med = med_df["MEDICOINTERV1_ENCRIPTADO"].nunique()
        top5_volume = med_df.head(5)["count"].sum()
        total_volume = med_df["count"].sum()
        concentracion = (top5_volume / total_volume) * 100
        
        print(f"✓ {año}: {num_med} médicos intervinientes distintos. "
              f"Top 5 concentra {concentracion:.1f}% de episodios.")

✓ 2019: 9617 médicos intervinientes distintos. Top 5 concentra 9.0% de episodios.
✓ 2020: 10388 médicos intervinientes distintos. Top 5 concentra 8.5% de episodios.
✓ 2021: 9476 médicos intervinientes distintos. Top 5 concentra 7.4% de episodios.
✓ 2022: 10198 médicos intervinientes distintos. Top 5 concentra 7.0% de episodios.
✓ 2023: 10569 médicos intervinientes distintos. Top 5 concentra 4.8% de episodios.
✓ 2024: 11164 médicos intervinientes distintos. Top 5 concentra 4.5% de episodios.


In [12]:
for año in ["2019", "2020", "2021"]: # Médicos intervinientes más frecuentes entre 2019 y 2021
    max_hosp = top5_med_interv_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_interv_por_año[año].to_string(index=False))


Año 2019 (Médico con mayor cantidad de intervenciones: 29905 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
                 11769.0  29905
                  2657.0   9568
                  9016.0   6245
                    54.0   5961
                 11780.0   2514

Año 2020 (Médico con mayor cantidad de intervenciones: 19202 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
                 11780.0  19202
                 11769.0   9290
                  2657.0   3601
                  9016.0   1844
                  6667.0   1077

Año 2021 (Médico con mayor cantidad de intervenciones: 28198 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  28198
              68823337.0   1093
              68804909.0   1004
              70326745.0    947
              77772744.0    928


In [13]:
for año in ["2022", "2023", "2024"]: # Médicos intervinientes más frecuentes entre 2022 y 2024  
    max_hosp = top5_med_interv_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_interv_por_año[año].to_string(index=False))


Año 2022 (Médico con mayor cantidad de intervenciones: 32947 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  32947
              71043147.0   1230
              70326745.0   1150
              70517922.0   1149
              77772744.0   1053

Año 2023 (Médico con mayor cantidad de intervenciones: 23200 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  23200
              70517922.0   1598
              69957946.0   1491
              80800383.0   1177
              80847393.0   1150

Año 2024 (Médico con mayor cantidad de intervenciones: 22221 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  22221
              71043147.0   1821
              70517922.0   1619
              68804909.0   1155
              71057175.0   1118


4. **MEDICOALTA_ENCRIPTADO**
- **Tipo de variable:** Hospitalaria (identificador).
- **Descripción:** Código encriptado que identifica al médico tratante que firma formalmente el egreso del paciente.
- **Veredicto:** Descartar. Al igual que el médico interviniente, su inclusión carece de justificación epidemiológica e introduce un ruido inmanejable para los selectores de características. Posee una cardinalidad extrema (hasta 17.743 médicos distintos en 2024) y está fuertemente contaminada por códigos genéricos de servicio (el ID `98438434` firmó más de 97.200 altas en 2022). Transformar esta variable es computacionalmente ineficiente y desviaría al modelo de su objetivo principal: descubrir perfiles clínicos e institucionales a nivel de red sanitaria.

In [14]:
# ============================================================================
# SECCIÓN 4: ANÁLISIS DE MÉDICOS DE ALTA (MEDICOALTA_ENCRIPTADO)
# ============================================================================
ruta_resultados_med_alta = "../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta"
os.makedirs(ruta_resultados_med_alta, exist_ok=True)

top5_med_alta_por_año = {}

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    año = archivo[-8:-4]
    
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    if "MEDICOALTA_ENCRIPTADO" in df.columns:
        df["MEDICOALTA_ENCRIPTADO"] = df["MEDICOALTA_ENCRIPTADO"].astype(str)
        df = df[~df["MEDICOALTA_ENCRIPTADO"].isin(["SIN INFORMACION", "DESCONOCIDO", "nan", "None"])]
        
        conteo_med = df["MEDICOALTA_ENCRIPTADO"].value_counts()
        med_df = conteo_med.reset_index()
        med_df.columns = ["MEDICOALTA_ENCRIPTADO", "count"]
        
        ruta_salida = f"{ruta_resultados_med_alta}/medicos_alta_{año}.csv"
        med_df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
        top5_med_alta_por_año[año] = med_df.head(5)
        
        num_med = med_df["MEDICOALTA_ENCRIPTADO"].nunique()
        top5_volume = med_df.head(5)["count"].sum()
        total_volume = med_df["count"].sum()
        concentracion = (top5_volume / total_volume) * 100
        
        print(f"✓ {año}: {num_med} médicos de alta distintos. "
              f"Top 5 concentra {concentracion:.1f}% de episodios.")

✓ 2019: 14104 médicos de alta distintos. Top 5 concentra 14.5% de episodios.
✓ 2020: 16051 médicos de alta distintos. Top 5 concentra 12.4% de episodios.
✓ 2021: 15500 médicos de alta distintos. Top 5 concentra 11.2% de episodios.
✓ 2022: 15720 médicos de alta distintos. Top 5 concentra 11.2% de episodios.
✓ 2023: 16479 médicos de alta distintos. Top 5 concentra 8.3% de episodios.
✓ 2024: 17743 médicos de alta distintos. Top 5 concentra 6.5% de episodios.


In [15]:
for año in ["2019", "2020", "2021"]: # Médicos de alta más frecuentes entre 2019 y 2021
    max_hosp = top5_med_alta_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico de alta con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_alta_por_año[año].to_string(index=False))


Año 2019 (Médico de alta con mayor cantidad de intervenciones: 91808 intervenciones):
MEDICOALTA_ENCRIPTADO  count
               2932.0  91808
              16856.0  33992
                744.0  17905
              10081.0  15789
               8068.0   7099

Año 2020 (Médico de alta con mayor cantidad de intervenciones: 56847 intervenciones):
MEDICOALTA_ENCRIPTADO  count
               2945.0  56847
               2932.0  23391
              16856.0   8792
                744.0   4634
              10081.0   3101

Año 2021 (Médico de alta con mayor cantidad de intervenciones: 84036 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  84036
             70572273   1999
             68804909   1837
             73512817   1649
             73765685   1644


In [15]:
for año in ["2022", "2023", "2024"]: # Médicos de alta más frecuentes entre 2022 y 2024
    max_hosp = top5_med_alta_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico de alta con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_alta_por_año[año].to_string(index=False))


Año 2022 (Médico de alta con mayor cantidad de intervenciones: 97239 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  97239
             98438441   1890
             98195735   1778
             76891172   1696
             77772744   1521

Año 2023 (Médico de alta con mayor cantidad de intervenciones: 77100 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  77100
             98438441   3196
             71851766   1949
             73765685   1880
             68804909   1852

Año 2024 (Médico de alta con mayor cantidad de intervenciones: 62211 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  62211
            102211195   2524
             77427459   2125
             71043147   1821
             68804909   1804
